# 1. IMPORT LIBRARIES

In [1]:
import pandas as pd

# 2. LOAD RAW DATA

In [36]:
df = pd.read_excel('explosive_weapon_raw.xlsx')

# 3. CLEAN COLUMN NAMES

In [ ]:
# Convert column names to lowercase, remove extra spaces, and replace spaces/slashes with underscores.
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(' ', '_')
    .str.replace('/', '_')
)

# Check result
print(df.columns.tolist())

['date', 'event_description', 'country', 'country_iso', 'admin_1', 'latitude', 'longitude', 'geo_precision', 'sector_affected', 'provider', 'launch_type', 'explosive_weapon_type', 'reported_perpetrator', 'reported_perpetrator_name', 'affected', 'food_systems_damaged_destroyed', 'water_systems_damaged_destroyed', 'aid_infrastructure_damaged_destroyed', 'health_infrastructure_damaged_destroyed', 'education_infrastructure_damaged_destroyed', 'idp_refugee_camp_building', 'aid_workers_killed', 'health_workers_killed', 'aid_health_workers_killed', 'educators_killed', 'students_killed', 'sind_event_id', 'date_event_entered', 'date_event_modified']


# 4. FILTER DATASET TO UKRAINE AND FULL-SCALE INVASION PERIOD

In [38]:
ukraine_df=df[
    (df['country_iso']=='UKR') &
    (df['date']>='2022-02-24')
    ].copy()

# 5. BASIC SCOPE VALIDATION

In [63]:
print('First reported date:', ukraine_df['date'].min())
print('Last reported date', ukraine_df['date'].max())
print('Countries:', ukraine_df['country_iso'].unique())
ukraine_df['sind_event_id'].nunique() == len(ukraine_df)

# Check whether each row represents a unique incident
print(
    "Unique event IDs:",
    ukraine_df["sind_event_id"].nunique() == len(ukraine_df)
)

First reported date: 2022-02-24 00:00:00
Last reported date 2026-07-21 00:00:00
Countries: <StringArray>
['UKR']
Length: 1, dtype: str
Unique event IDs: True


# 6. REMOVE COMPLETELY EMPTY COLUMNS

In [ ]:
empty_columns = ukraine_df.columns[
    ukraine_df.isna().all()
]

print('Completely empty columns:')
print(empty_columns.tolist())

ukraine_df = ukraine_df.drop(
    columns=empty_columns
)

Completely empty columns:
[]


# 7. REGION CLEANING

In [ ]:
# 1. Preserve raw geography and normalize text
ukraine_df['region_clean'] = (
    ukraine_df['admin_1']
    .str.strip()
    .str.lower()
)

# QA columns
ukraine_df['region_qa_status'] = 'ok'
ukraine_df['region_qa_note'] = ''

# Remember source-level unknown regions BEFORE replacement
source_unknown_mask = (
    ukraine_df['region_clean'] == 'no information'
)

# 7.1 Safe region normalization

In [ ]:
region_mapping = {
    'zaporizhzhia oblast': 'zaporizhia oblast',
    'oblast de kharkiv': 'kharkiv oblast',

    'khust raion': 'zakarpattia oblast',
    'zolochiv raion': 'lviv oblast',
    'beryslav raion': 'kherson oblast',
    'kherson raion': 'kherson oblast',
    'henichesk raion': 'kherson oblast',

    'pecherskyi': 'kyiv city',
    'pecherskyi district': 'kyiv city',
    'sviatoshynskyi district': 'kyiv city',
    'obolonskyi district': 'kyiv city',
    'holosiivskyi district': 'kyiv city',
    'podilskyi district': 'kyiv city',
    'darnytskyi district': 'kyiv city',
    'dniprovskyi district': 'kyiv city',
    'solomianskyi district': 'kyiv city',
    'desnianskyi district': 'kyiv city',

    'no information': 'unknown',

    'autonomous republic of crimea': 'crimea',
    'republic of crimea': 'crimea',
}

ukraine_df['region_clean'] = (
    ukraine_df['region_clean']
    .replace(region_mapping)
)

ukraine_df['region_clean'].value_counts()


region_clean
donetsk oblast              902
kharkiv oblast              785
kherson oblast              731
dnipropetrovsk oblast       662
sumy oblast                 446
unknown                     252
chernihiv oblast            195
zaporizhia oblast           194
mykolaiv oblast             163
odesa oblast                138
kyiv oblast                 124
kyiv city                    91
luhansk oblast               88
shevchenkivskyi district     47
poltava oblast               29
lviv oblast                  26
cherkasy oblast              20
zhytomyr oblast              20
vinnytsia oblast             14
kirovohrad oblast            13
ivano-frankivsk oblast       10
khmelnytskyi oblast           8
rivne oblast                  7
crimea                        6
chernivtsi oblast             6
ternopil oblast               6
volyn oblast                  4
zakarpattia oblast            2
Name: count, dtype: int64

# 7.2 Mark source-level unknown regions

In [69]:
ukraine_df.loc[
    source_unknown_mask,
    "region_qa_status"
] = "review"

ukraine_df.loc[
    source_unknown_mask,
    "region_qa_note"
] = "Region reported as No Information in source data."

# 7.3 Manually verified event-level corrections

In [70]:
# Incidents manually verified using external reporting, due to an ambigous Admin 1 value 
region_corrections = {
    135045: 'kyiv city',
    134975: 'kyiv city',
    134980: 'kyiv city',
    134978: 'kyiv city',
    134977: 'kyiv city',
    134976: 'kyiv city',
    134979: 'kyiv city',
    134974: 'kyiv city',
    134973: 'kyiv city',
    134972: 'kyiv city',
    134971: 'kyiv city',
    134963: 'kyiv city',

    133301: 'kyiv city',
    133039: 'kharkiv oblast',

    124669: 'kyiv city',
    124726: 'kyiv city',
    124905: 'kyiv city',

    121116: 'kyiv city',
    120373: 'kyiv city',
    120376: 'kyiv city',

    113362: 'kyiv city',

    118835: 'kyiv city',
    118839: 'kyiv city',
    118841: 'kyiv city',

    115771: 'kharkiv oblast',
    118830: 'kharkiv oblast',

    45551: 'kyiv city',
    45552: 'kyiv city',
    45555: 'kyiv city',
    45556: 'kyiv city',
    45557: 'kyiv city',
    45559: 'kyiv city',

    84500: 'kharkiv oblast',
    39111: 'kyiv city',
}

ukraine_df['region_correction'] = (
    ukraine_df['sind_event_id']
    .map(region_corrections)
)

corrected_mask = (
    ukraine_df['region_correction'].notna()
)

ukraine_df.loc[
    corrected_mask,
    'region_clean'
] = ukraine_df.loc[
    corrected_mask,
    'region_correction'
]

ukraine_df.loc[
    corrected_mask,
    'region_qa_status'
] = 'corrected'

ukraine_df.loc[
    corrected_mask,
    'region_qa_note'
] = (
    'Region manually verified and corrected '
    'using external reporting.'
)

# 7.4 Handle unresolved Shevchenkivskyi District

In [71]:
ambiguous_mask = (
    ukraine_df["region_clean"]
    == "shevchenkivskyi district"
)

ukraine_df.loc[
    ambiguous_mask,
    "region_qa_status"
] = "review"

ukraine_df.loc[
    ambiguous_mask,
    "region_qa_note"
] = (
    "Ambiguous Shevchenkivskyi district; "
    "city could not be reliably determined "
    "from the source data."
)

ukraine_df.loc[
    ambiguous_mask,
    "region_clean"
] = "unknown"

# 7.5 Handle missing Admin 1

In [72]:
missing_region_mask = (
    ukraine_df["region_clean"].isna()
)

ukraine_df.loc[
    missing_region_mask,
    "region_clean"
] = "unknown"

ukraine_df.loc[
    missing_region_mask,
    "region_qa_status"
] = "review"

ukraine_df.loc[
    missing_region_mask,
    "region_qa_note"
] = "Region is missing in the source data."


# Temporary correction field is no longer needed
ukraine_df = ukraine_df.drop(
    columns=["region_correction"]
)


# 7.6 Region whitelist validation

In [73]:
expected_regions = {
    "vinnytsia oblast",
    "volyn oblast",
    "dnipropetrovsk oblast",
    "donetsk oblast",
    "zhytomyr oblast",
    "zakarpattia oblast",
    "zaporizhia oblast",
    "ivano-frankivsk oblast",
    "kyiv oblast",
    "kirovohrad oblast",
    "luhansk oblast",
    "lviv oblast",
    "mykolaiv oblast",
    "odesa oblast",
    "poltava oblast",
    "rivne oblast",
    "sumy oblast",
    "ternopil oblast",
    "kharkiv oblast",
    "kherson oblast",
    "khmelnytskyi oblast",
    "cherkasy oblast",
    "chernivtsi oblast",
    "chernihiv oblast",
    "kyiv city",
    "crimea",
    "unknown",
}

actual_regions = set(
    ukraine_df["region_clean"]
    .dropna()
    .unique()
)

unexpected_regions = (
    actual_regions - expected_regions
)

if unexpected_regions:
    raise ValueError(
        f"Unexpected region values found: "
        f"{unexpected_regions}"
    )


# Check geography result
print(
    ukraine_df["region_clean"]
    .value_counts(dropna=False)
)

print(
    ukraine_df["region_qa_status"]
    .value_counts(dropna=False)
)

region_clean
donetsk oblast            902
kharkiv oblast            789
kherson oblast            731
dnipropetrovsk oblast     662
sumy oblast               446
unknown                   298
chernihiv oblast          195
zaporizhia oblast         194
mykolaiv oblast           163
odesa oblast              138
kyiv oblast               124
kyiv city                 121
luhansk oblast             88
poltava oblast             29
lviv oblast                26
cherkasy oblast            20
zhytomyr oblast            20
vinnytsia oblast           14
kirovohrad oblast          13
ivano-frankivsk oblast     10
khmelnytskyi oblast         8
rivne oblast                7
crimea                      6
chernivtsi oblast           6
ternopil oblast             6
volyn oblast                4
zakarpattia oblast          2
Name: count, dtype: int64
region_qa_status
ok           4690
review        298
corrected      34
Name: count, dtype: int64


# 8. CLEAN EXPLOSIVE WEAPON TYPES

In [75]:
# Preserve raw values and create cleaned analytical field
ukraine_df['explosive_weapon_type_clean'] = (
    ukraine_df['explosive_weapon_type']
    .str.strip()
    .str.lower()
)

# Normalize equivalent categories
replace_explosive = {
    'aerial bomb, unspecified explosive': 'aerial bomb',
    'missile, unspecified explosive': 'missile',
}

ukraine_df['explosive_weapon_type_clean'] = (
    ukraine_df['explosive_weapon_type_clean']
    .replace(replace_explosive)
    .str.title()
)

# Restore correct capitalization for acronyms
acronym_mapping = {
    'Uxo': 'UXO',
    'Rpg': 'RPG',
    'Svied': 'SVIED',
    'Unspecified Ied': 'Unspecified IED',
}

ukraine_df['explosive_weapon_type_clean'] = (
    ukraine_df['explosive_weapon_type_clean']
    .replace(acronym_mapping)
)

# Check result
ukraine_df['explosive_weapon_type_clean'].value_counts(dropna=False)

explosive_weapon_type_clean
Aerial Bomb              2481
Shelling                  745
Unspecified Explosive     617
Missile                   500
Artillery                 281
Rocket                    259
Cluster Bomb               50
Mortar                     39
Mine                       35
UXO                         5
Hand Grenade                4
Unspecified IED             4
RPG                         1
SVIED                       1
Name: count, dtype: int64

# 9. CREATE DAMAGE INDICATORS

In [ ]:
# These source fields are sparse.
# A non-null value is currently treated as evidence
# that the corresponding system was damaged.
ukraine_df['food_systems_damage_count'] = ukraine_df['food_systems_damaged_destroyed'].notna().astype(int)
ukraine_df['water_systems_damage_count'] = ukraine_df['water_systems_damaged_destroyed'].notna().astype(int)

# 10. CALCULATE INFRASTRUCTURE TOTAL

In [77]:
infrastructure_cols = [
    'food_systems_damage_count',
    'water_systems_damage_count',
    'aid_infrastructure_damaged_destroyed',
    'health_infrastructure_damaged_destroyed',
    'education_infrastructure_damaged_destroyed',
    'idp_refugee_camp_building',
]

# Missing values are treated as 0 only during calculation.
# Raw source fields remain unchanged.
ukraine_df['infrastructure_total'] = (
    ukraine_df[infrastructure_cols]
    .fillna(0)
    .sum(axis=1)
)

# 11. CALCULATE RECORDED DEATHS

In [78]:
# Calculate total deaths per incident
death_cols=[
    'aid_workers_killed',
    'health_workers_killed', 
    'aid_health_workers_killed',
    'educators_killed', 
    'students_killed'
]

ukraine_df['recorded_deaths'] = (
    ukraine_df[death_cols]
    .fillna(0)
    .sum(axis=1)
)

# 12. CREATE SECTOR TABLE

In [79]:
# Split sectors and make a new table for proper KPI count
sector_df = ukraine_df.copy()

sector_df['sector_clean']= (
    sector_df['sector_affected']
    .fillna('Unknown')
    .str.strip()
    .str.split(', ')
)

sector_df = sector_df.explode('sector_clean')


sector_df['sector_clean'] = sector_df['sector_clean'].str.strip()

sector_df['sector_clean'].value_counts(dropna=False)

sector_clean
Health Care       2401
Education         1561
Food Systems       897
Aid Operations     219
Water Systems      134
Protection           7
Unknown              1
Name: count, dtype: int64

# 13. FINAL DATA QUALITY CHECKS

In [80]:
print(
    "Main table shape:",
    ukraine_df.shape
)

print(
    "Sector table shape:",
    sector_df.shape
)

print("\nMissing values in key columns:")

print(
    ukraine_df[
        [
            "date",
            "sind_event_id",
            "region_clean",
            "recorded_deaths",
            "infrastructure_total",
        ]
    ]
    .isna()
    .sum()
)

# Main table should still contain one row per event
assert (
    ukraine_df["sind_event_id"].nunique()
    == len(ukraine_df)
)

# Exploding sectors must not remove incidents
assert (
    sector_df["sind_event_id"].nunique()
    == ukraine_df["sind_event_id"].nunique()
)

print("\nTransform completed successfully.")

Main table shape: (5022, 34)
Sector table shape: (5220, 35)

Missing values in key columns:
date                    0
sind_event_id           0
region_clean            0
recorded_deaths         0
infrastructure_total    0
dtype: int64

Transform completed successfully.
